<a href="https://colab.research.google.com/github/dnriski/data-science-2026/blob/main/Pertemuan3_RiskiDwiNugroho_230401010213.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
# 2. Import Data
import requests, pandas as pd
import numpy as np
from scipy.stats import mstats
from pandas import json_normalize

In [50]:
# 3. Muat Dataset dari path lokal
df = pd.read_csv("sample_data/housing_dirty.csv")

# 4. Eksplorasi Awal
## df.info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [51]:
## df.describe()
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [52]:
## df.isnull().sum()
df.isnull().sum()

,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


In [53]:
# 5. Hapus Duplikat dengan df.drop_duplicates(inplace=True)
df.drop_duplicates(inplace=True)
print("Hasil Penghapusan", df.shape)

Hasil Penghapusan (130, 7)


In [54]:
# 6. Normalisasi string
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

In [55]:
# 7. Imputasi missing values
df ['luas']= df['luas_m2'].fillna(df['luas_m2'].mean())
mode_kota=df['kota'].mode()[0]
df['kota']=df['kota'].fillna(mode_kota)
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [56]:
# 8. Tangani Outlier dengan IQR Fence
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
  Q1, Q3 = df[col].quantile([0.25, 0.75])
  IQR = Q3 - Q1
  df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

In [57]:
# 9. Validasi Akhir
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)

# 10. Ekspor df.to_csv
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Shape akhir: (130, 8)
Dataset bersih tersimpan!


In [58]:
# 11. Akses API Json & simpan respons
URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

# 2 Selalu cek status code terlebih dahulu
if response.status_code == 200:
  data = response.json()
  df = json_normalize(data, sep='_')
  df_api = pd.DataFrame(data)
  print("Berhasil mengambil data")
  print(df_api.head())
else:
  print("gagal")

Berhasil mengambil data
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

         website                                            company  
0  hildegard.org  {'name': 'Romaguera-Cr